In [ ]:
!pip install langchain langchain_core langchain-community langchain-groq gradio python-dotenv
!pip install openpyxl langchain-huggingface faiss-cpu pypdf sentence-transformers -q
import pandas as pd

faq_data = {
    "Category": [
        "Program Overview", "Program Structure", "Program Structure",
        "Pricing & Fees", "Pricing & Fees", "Curriculum & Skills",
        "Curriculum & Skills", "Evaluation & Projects", "Career & Placement",
        "Leadership & Contact"
    ],
    "Question": [
        "What is the total duration and structure of the PragyanAI program?",
        "What happens in Phase 1 (First 6 Months)?",
        "What happens in Phase 2 (12 Months)?",
        "What is the fee structure for the Founding Batch?",
        "What is the salary potential after completing the program?",
        "What modules are covered in Months 1-3 (Foundational Core)?",
        "What modules are covered in Months 4-6 (Advanced Frontier)?",
        "How are students evaluated during the 6-month offline training?",
        "What career tracks or roles are unlocked?",
        "Who leads PragyanAI and how can I contact them?"
    ],
    "Answer": [
        "The PragyanAI AI GenAI program is an 18-month journey comprising 6 Months of Fully Offline Training followed by a 12-Month Internship & Placement Drive.",
        "Phase 1 (6 Months) consists of intensive offline training with half-day classroom sessions, half-day hands-on labs, real-time projects, monthly hackathons, and technical seminars.",
        "Phase 2 (12 Months) includes an extended internship, live client assignments, technical mock interviews, resume building, and startup/product development exposure.",
        "Founding Batch (First 100 students): Initial Training Fee is ₹50,000 + Success Fee of ₹50,000 after placement (Total ₹1,00,000, discounted from standard ₹1,50,000).",
        "Target packages: AI Engineer (₹6–₹15 LPA), GenAI Engineer (₹8–₹18 LPA), and Agentic AI Engineer (₹10–₹25 LPA).",
        "Month 1: Python Full Stack & Analytics. Month 2: Data Science & BI Analytics. Month 3: Machine Learning Frameworks (AutoML, Streamlit deployment).",
        "Month 4: Deep Learning & Computer Vision (CNNs, PyTorch, YOLO). Month 5: NLP & Generative AI (LLMs, RAG, LangChain, Fine-tuning). Month 6: Agentic AI (CrewAI, AutoGen, Multi-Agent Systems, MCP).",
        "Students participate in 1 Technical Seminar per skill (evaluated out of 100 marks) and 1 Skill-wise 48-Hour Hackathon with cash prizes (₹5,000 winner, ₹3,000 runner-up).",
        "7 Multi-Track Pathways: Data Analyst, Data Scientist & ML, AI Engineer, GenAI Engineer, Agentic AI Engineer, Product/MVP Engineer, and Software Engineer.",
        "Led by Sateesh Ambesange (Co-Founder, NITK alumnus, 25+ years IT exp). Phone: +91-9741007422 | Email: sateesh.ambesange@pragyanai.com / pragyan.ai.school@gmail.com"
    ]
}

df = pd.DataFrame(faq_data)
df.to_excel("pragyan_faq_prices.xlsx", index=False)
print("✅ Created 'pragyan_faq_prices.xlsx' with PragyanAI presentation data!")
import os
import pandas as pd
import gradio as gr
from langchain_community.document_loaders import PyPDFLoader
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.output_parsers import StrOutputParser
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_groq import ChatGroq
# ---------------------------------------------------------------------------
# 1. System Prompts specifically grounded in PragyanAI Data
# ---------------------------------------------------------------------------
SALES_PROMPTS = {
    "PragyanAI Student Counselor": """You are Aarav, an Academic & Career Advisor for PragyanAI.
Goal: Guide prospective students to enroll in the 18-Month AI/GenAI Program (6 Month Offline Training + 12 Month Placement Drive).

Strict Rule: Answer pricing, fee structures, curriculum details, and salary potential ONLY based on the Document Context below.

Retrieved Document Context:
{context}

Behavior Guidelines:
1. Be encouraging, empathetic, and focus on practical "builder" skill transformation.
2. Highlight key advantages: 100+ projects, 48-hour hackathons, risk-shared pricing (pay-after-placement success fee), and direct mentorship under Sateesh Ambesange.""",

    "PragyanAI Institutional / CoE Advisor": """You are Dr. Kavita, Institutional Relations Lead at PragyanAI.
Goal: Partner with engineering colleges to solve the education trap and transform students from theory learners into product builders.

Strict Rule: Use the retrieved Context below to cite exact program structures, multi-track career pathways, and evaluation rubrics (seminars, hackathons).

Retrieved Document Context:
{context}

Behavior Guidelines:
1. Maintain an authoritative, industry-oriented tone.
2. Focus on bridging the gap between college curricula and high-value industry roles (Agentic AI, GenAI).""",

    "PragyanAI Enterprise AI & Placement Lead": """You are Rohan, Enterprise Placement & Venture Lead at PragyanAI.
Goal: Connect hiring partners and enterprise leaders with top-tier PragyanAI builders and discuss talent recruitment or custom AI automation.

Strict Rule: Reference exact technical skills (CrewAI, AutoGen, LangChain, RAG, Multi-Agent systems) and portfolio deliverables (GitHub profile, live deployed MVPs) from the context below.

Retrieved Document Context:
{context}

Behavior Guidelines:
1. Confident, direct, and ROI-driven tone.
2. Emphasize that PragyanAI engineers are class-hired builders capable of deploying live applications immediately."""
}
# ---------------------------------------------------------------------------
# 2. Vector Store Indexer (Loads Excel FAQ + PDF Documents)
# ---------------------------------------------------------------------------
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
vectorstore = None

def load_documents_into_vectorstore(file_paths=None):
    global vectorstore
    docs = []

    # 1. Process UI file uploads (PDFs or Excel files)
    if file_paths:
        for file in file_paths:
            path = file.name if hasattr(file, 'name') else file
            if path.endswith('.pdf'):
                loader = PyPDFLoader(path)
                docs.extend(loader.load())
            elif path.endswith('.xlsx') or path.endswith('.xls'):
                excel_df = pd.read_excel(path)
                for _, row in excel_df.iterrows():
                    content = " | ".join([f"{col}: {val}" for col, val in row.items()])
                    docs.append(Document(page_content=content, metadata={"source": path}))

    # 2. Automatically load default Excel FAQ if present locally
    if os.path.exists("pragyan_faq_prices.xlsx"):
        excel_df = pd.read_excel("pragyan_faq_prices.xlsx")
        for _, row in excel_df.iterrows():
            content = " | ".join([f"{col}: {val}" for col, val in row.items()])
            docs.append(Document(page_content=content, metadata={"source": "pragyan_faq_prices.xlsx"}))

    # Fallback knowledge base if no files are loaded
    if not docs:
        docs = [
            Document(page_content="PragyanAI Program: 6 Months Offline Training + 12 Months Placement Drive. Led by Sateesh Ambesange."),
            Document(page_content="Founding Batch Fee: ₹50,000 initial training + ₹50,000 success fee post placement.")
        ]

    vectorstore = FAISS.from_documents(docs, embeddings)
    return f"✅ PragyanAI Knowledge Base updated successfully with {len(docs)} document chunks!"

# Build initial index
load_documents_into_vectorstore()
from google.colab import userdata
# Retrieve your key
groq_api_key = userdata.get('GROQ_API_KEY')


# ---------------------------------------------------------------------------
# 3. Groq LLM & LCEL RAG Pipeline
# ---------------------------------------------------------------------------

llm = ChatGroq(
    groq_api_key=groq_api_key,
    model_name="llama-3.3-70b-versatile",
    temperature=0.3
)

store = {}

def get_session_history(session_id: str):
    if session_id not in store:
        store[session_id] = ChatMessageHistory()
    return store[session_id]

def create_rag_chain(persona_name: str, retrieved_context: str):
    system_instruction = SALES_PROMPTS.get(
        persona_name,
        SALES_PROMPTS["PragyanAI Student Counselor"]
    ).format(context=retrieved_context)

    prompt = ChatPromptTemplate.from_messages([
        ("system", system_instruction),
        MessagesPlaceholder(variable_name="history"),
        ("human", "{input}")
    ])

    return prompt | llm | StrOutputParser()
# ---------------------------------------------------------------------------
# 4. Gradio Callbacks
# ---------------------------------------------------------------------------
def respond(message, history, persona_name):
    if not message.strip():
        return ""

    # Search top relevant snippets
    retriever = vectorstore.as_retriever(search_kwargs={"k": 4})
    relevant_docs = retriever.invoke(message)
    context_str = "\n".join([f"- {doc.page_content}" for doc in relevant_docs])

    session_id = f"pragyan_session_{persona_name.replace(' ', '_')}"
    base_chain = create_rag_chain(persona_name, context_str)

    conversational_chain = RunnableWithMessageHistory(
        base_chain,
        get_session_history,
        input_messages_key="input",
        history_messages_key="history",
    )

    return conversational_chain.invoke(
        {"input": message},
        config={"configurable": {"session_id": session_id}}
    )

def clear_chat_history(persona_name):
    session_id = f"pragyan_session_{persona_name.replace(' ', '_')}"
    if session_id in store:
        store[session_id].clear()
# ---------------------------------------------------------------------------
# 5. Gradio User Interface
# ---------------------------------------------------------------------------
with gr.Blocks(title="PragyanAI Intelligent Assistant") as demo:
    gr.Markdown("# 🎓 PragyanAI Conversational Sales & FAQ Assistant")
    gr.Markdown("Answers program questions based on the **PragyanAI Presentation & FAQ Sheet**.")

    with gr.Row():
        with gr.Column(scale=1):
            persona_selector = gr.Dropdown(
                choices=list(SALES_PROMPTS.keys()),
                value="PragyanAI Student Counselor",
                label="Select PragyanAI Persona",
                interactive=True
            )
            file_uploader = gr.File(
                label="Upload Additional PDFs or Excel Sheets",
                file_count="multiple",
                file_types=[".pdf", ".xlsx", ".xls"]
            )
            upload_status = gr.Textbox(label="Knowledge Base Status", value="PragyanAI presentation FAQ pre-loaded.", interactive=False)
            file_uploader.change(fn=load_documents_into_vectorstore, inputs=[file_uploader], outputs=[upload_status])

        with gr.Column(scale=3):
            chatbot_ui = gr.ChatInterface(
                fn=respond,
                additional_inputs=[persona_selector]
            )
            clear_btn = gr.Button("Clear Memory for Selected Persona", variant="secondary")
            clear_btn.click(fn=clear_chat_history, inputs=[persona_selector], outputs=None)

if __name__ == "__main__":
    demo.launch(share=True, debug=True)

✅ Created 'pragyan_faq_prices.xlsx' with PragyanAI presentation data!


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://31d8d25263e86c8b86.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


/usr/local/lib/python3.12/dist-packages/anyio/_backends/_asyncio.py:1033: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  result = context.run(func, *args)
/usr/local/lib/python3.12/dist-packages/anyio/_backends/_asyncio.py:1033: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  result = context.run(func, *args)


In [ ]:
CSV_FILE = "candidate_data.csv"
if not os.path.exists(CSV_FILE):
    with open(CSV_FILE, mode="w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow(["Name", "Email", "Phone", "Course Interest"])

SALES_PROMPTS = {
    "PragyanAI Student Counselor": """You are Aarav, an Academic & Career Advisor for PragyanAI.
Goal: Guide prospective students to enroll in the 18-Month AI/GenAI Program (6 Month Offline Training + 12 Month Placement Drive).

Strict Rule: Answer pricing, fee structures, curriculum details, and salary potential based on the Document Context and Search Results below.

Retrieved Context:
{context}

Behavior Guidelines:
1. Be encouraging, empathetic, and focus on practical "builder" skill transformation.
2. Highlight key advantages: 100+ projects, 48-hour hackathons, risk-shared pricing (pay-after-placement success fee), and direct mentorship under Sateesh Ambesange.""",

    "PragyanAI Institutional / CoE Advisor": """You are Dr. Kavita, Institutional Relations Lead at PragyanAI.
Goal: Partner with engineering colleges to solve the education trap and transform students into product builders.

Retrieved Context:
{context}""",

    "PragyanAI Enterprise AI & Placement Lead": """You are Rohan, Enterprise Placement & Venture Lead at PragyanAI.
Goal: Connect hiring partners and enterprise leaders with top-tier PragyanAI builders.

Retrieved Context:
{context}"""
}

LANGUAGES = {
    "English": "en",
    "Hindi": "hi",
    "Kannada": "kn",
    "Telugu": "te",
    "Tamil": "ta",
    "Spanish": "es",
    "French": "fr"
}

vector_store = None
raw_extracted_text = ""
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
uploaded_file_paths = {}  # Map: file_name -> path

Sateesh Ambesange
10:53
def initialize_default_knowledge_base():
    global vector_store, raw_extracted_text

    # Auto-load default FAQ Excel if available locally
    if os.path.exists("pragyan_faq_prices.xlsx"):
        import pandas as pd
        excel_df = pd.read_excel("pragyan_faq_prices.xlsx")
        text_content = excel_df.to_string()
        raw_extracted_text = text_content

        from langchain_core.documents import Document
        doc = Document(page_content=text_content, metadata={"source": "pragyan_faq_prices.xlsx"})

        text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=100)
        chunks = text_splitter.split_documents([doc])
        vector_store = FAISS.from_documents(chunks, embeddings)
        print("✅ Pre-loaded default PragyanAI knowledge base.")

# Auto-run on startup
initialize_default_knowledge_base()
# ==============================================================================
# 3. SERPAPI LIVE WEB SEARCH MODULE
# ==============================================================================
def google_search_api(query, num_results=3):
    """Executes live web search using SerpAPI."""
    api_key = userdata.get('SERPAPI_API_KEY') or serpapi_key

    if not api_key:
        return "[SERPAPI_API_KEY not configured in environment variables]"

    try:
        params = {
            "engine": "google",
            "q": query,
            "num": num_results,
            "api_key": api_key,
        }

        search = GoogleSearch(params)
        results_dict = search.get_dict()

        organic_results = results_dict.get("organic_results", [])
        if not organic_results:
            return "No web results found."

        search_snippets = []
        for idx, item in enumerate(organic_results[:num_results], 1):
            title = item.get("title", "No Title")
            snippet = item.get("snippet", "No snippet available.")
            link = item.get("link", "")
            search_snippets.append(f"[{idx}] {title}\nSnippet: {snippet}\nURL: {link}")

        return "\n\n".join(search_snippets)

    except Exception as e:
        return f"[SerpAPI Search Error: {str(e)}]"
# ==============================================================================
# 4. MULTI-ENGINE PDF EXTRACTION & VECTOR STORE INDEXING
# ==============================================================================
def extract_text_multi_engine(file_path):
    """Attempts extraction using Docling OCR -> PyPDF -> PDFPlumber Fallback."""
    text_content = ""

    # Engine 1: IBM Docling (OCR & Layout Aware) - Uncomment if enabled
    # try:
    #     result = docling_converter.convert(file_path)
    #     text_content = result.document.export_to_markdown()
    #     if len(text_content.strip()) > 50:
    #         return text_content, "Docling OCR"
    # except Exception as e:
    #     print(f"Docling Engine skipped/failed: {e}")

    # Engine 2: PyPDF (Fast text extraction)
    try:
        loader = PyPDFLoader(file_path)
        docs = loader.load()
        text_content = "\n\n".join([d.page_content for d in docs])
        if len(text_content.strip()) > 50:
            return text_content, "PyPDF"
    except Exception as e:
        print(f"PyPDF Engine skipped/failed: {e}")

    # Engine 3: PDFPlumber Fallback (Accurate text/layout extraction)
    try:
        loader = PDFPlumberLoader(file_path)
        docs = loader.load()
        text_content = "\n\n".join([d.page_content for d in docs])
        if len(text_content.strip()) > 50:
            return text_content, "PDFPlumber"
    except Exception as e:
        print(f"PDFPlumber Engine failed: {e}")

    return "", "Failed"


def load_documents_into_vectorstore(files):
    global vector_store, raw_extracted_text, uploaded_file_paths

    # Return exactly 3 items to match Gradio outputs: (status, dropdown_update, file_path)
    if not files:
        return "No files uploaded.", gr.update(choices=[], value=None), None

    documents = []
    text_buffer = []
    file_names = []
    uploaded_file_paths.clear()

    for file_obj in files:
        file_path = file_obj.name if hasattr(file_obj, "name") else file_obj
        file_name = os.path.basename(file_path)
        file_names.append(file_name)
        uploaded_file_paths[file_name] = file_path

        try:
            if file_path.endswith(".pdf"):
                text_content, engine_used = extract_text_multi_engine(file_path)
                if text_content:
                    documents.append(
                        Document(
                            page_content=text_content,
                            metadata={"source": file_name, "engine": engine_used}
                        )
                    )
                    text_buffer.append(
                        f"--- Document: {file_name} (Parsed via {engine_used})---\n{text_content}"
                    )

            elif file_path.endswith((".xlsx", ".xls")):
                df = pd.read_excel(file_path)
                text_content = df.to_string()
                documents.append(
                    Document(
                        page_content=text_content,
                        metadata={"source": file_name, "engine": "Pandas"}
                    )
                )
                text_buffer.append(text_content)

        except Exception as e:
            return (
                f"❌ Error loading file {file_name}: {str(e)}",
                gr.update(choices=[], value=None),
                None
            )

    if not documents:
        return (
            "⚠️ Could not extract text from uploaded documents.",
            gr.update(choices=[], value=None),
            None
        )

    raw_extracted_text = "\n\n".join(text_buffer)

    text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=100)
    chunks = text_splitter.split_documents(documents)

    if vector_store is None:
        vector_store = FAISS.from_documents(chunks, embeddings)
    else:
        vector_store.add_documents(chunks)

    first_pdf_path = uploaded_file_paths.get(file_names[0], None)

    return (
        f"✅ Loaded {len(files)} file(s) ac
# ==============================================================================
# 5. RAG & VOICE SYNTHESIS ENGINE
# ==============================================================================
def respond(query, persona="PragyanAI Student Counselor", google_search=True, web_search=False, history=None):
    global vector_store

    if not query or not query.strip():
        return "Please ask a question."

    pdf_context = ""
    web_context = ""

    if vector_store is not None:
        try:
            retriever = vector_store.as_retriever(search_type="mmr", search_kwargs={"k": 5, "fetch_k": 12})
            relevant_docs = retriever.invoke(query)
            if relevant_docs:
                pdf_context = "\n---\n".join([doc.page_content for doc in relevant_docs])
        except Exception as e:
            print(f"Retrieval Error: {e}")

    if google_search or web_search:
        web_context = google_search_api(query, num_results=3)

    full_retrieved_context = ""
    if pdf_context:
        full_retrieved_context += f"--- PDF DOCUMENT CONTEXT ---\n{pdf_context}\n\n"
    if web_context:
        full_retrieved_context += f"--- GOOGLE WEB SEARCH RESULTS ---\n{web_context}"

    if not full_retrieved_context:
        full_retrieved_context = "No relevant context found in documents or web."

    prompt_template = SALES_PROMPTS.get(persona, SALES_PROMPTS["PragyanAI Student Counselor"])
    formatted_system_prompt = prompt_template.format(context=full_retrieved_context)

    try:
        messages = [
            SystemMessage(content=formatted_system_prompt),
            ("human", query),
        ]
        response = llm.invoke(messages)
        return response.content
    except Exception as e:
        return f"### 📑 Extracted Context:\n{full_retrieved_context}\n\n*(Error: {str(e)})*"

def process_voice_and_chat(user_text, voice_file, target_lang, persona, google_search, web_search):
    query = ""

    if voice_file is not None:
        transcription = stt_model.transcribe(voice_file, fp16=False)
        query = transcription.get("text", "").strip()

    if not query and user_text:
        query = user_text.strip()

    if not query:
        return "Please ask a question via voice or text input.", None

    lang_code = LANGUAGES.get(target_lang, "en")

    if lang_code != "en":
        try:
            processed_query = GoogleTranslator(source="auto", target="en").translate(query)
        except Exception:
            processed_query = query
    else:
        processed_query = query

    raw_response = respond(
        query=processed_query,
        persona=persona,
        google_search=google_search,
        web_search=web_search
    )

    if lang_code != "en":
        try:
            final_response = GoogleTranslator(source="auto", target=lang_code).translate(raw_response)
        except Exception:
            final_response = raw_response
    else:
        final_response = raw_response

    tts_audio_path = "response.mp3"
    try:
        tts = gTTS(text=final_response, lang=lang_code, slow=False)
        tts.save(tts_audio_path)
    except Exception:
        tts_audio_path = None

    return final_response, tts_audio_path

def save_candidate_data(name, email, phone, course):
    if not name or not email:
        return "⚠️ Please provide at least a Name and Email.", CSV_FILE
    with open(CSV_FILE, mode="a", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow([name, email, phone, course])
    return f"✅ Saved record for {name}!", CSV_FILE

def on_pdf_selected(paper_name):
    """Updates the PDF viewer when a file is selected from the dropdown."""
    if paper_name in uploaded_file_paths:
        return uploaded_file_paths[paper_name]
    return None

def chat_interface_respond(message, history, persona, google_search, web_search):
    return respond(message, persona, google_search, web_search, history)
!pip install PyMuPDF


import base64
import os
import fitz  # PyMuPDF
import gradio as gr
from PIL import Image


def get_pdf_page_count(pdf_path):
  """Gets total pages in a PDF file using PyMuPDF."""
  if not pdf_path or not os.path.exists(pdf_path):
    return 0
  try:
    doc = fitz.open(pdf_path)
    count = len(doc)
    doc.close()
    return count
  except Exception as e:
    print(f"Error reading PDF page count: {e}")
    return 0


def render_pdf_page_as_image(pdf_name, page_num):
  """Renders a specific PDF page as a PIL Image."""
  if not pdf_name or pdf_name not in uploaded_file_paths:
    return None, 1, 0, "**Page 0 of 0**"

  file_path = uploaded_file_paths[pdf_name]
  total_pages = get_pdf_page_count(file_path)

  if total_pages == 0:
    return None, 1, 0, "**Error loading PDF pages**"

  # Clamp page number within bounds
  page_num = max(1, min(page_num, total_pages))

  try:
    doc = fitz.open(file_path)
    # PyMuPDF page numbers are 0-indexed
    page = doc.load_page(page_num - 1)

    # Scale matrix for high quality rendering (2.0 = 2x zoom/clarity)
    pix = page.get_pixmap(matrix=fitz.Matrix(2.0, 2.0))
    img = Image.frombytes("RGB", [pix.width, pix.height], pix.samples)
    doc.close()

    page_label = f"**Page {page_num} of {total_pages}**"
    return img, page_num, total_pages, page_label

  except Exception as e:
    print(f"Error rendering page to image: {e}")
    return None, page_num, total_pages, f"**Error loading page {page_num}**"


def navigate_pdf(pdf_name, current_page, action):
  """Handles Next, Previous, Start, End, and Direct Page Jumps."""
  if not pdf_name or pdf_name not in uploaded_file_paths:
    return None, 1, "**Page 0 of 0**"

  file_path = uploaded_file_paths[pdf_name]
  total_pages = get_pdf_page_count(file_path)

  if action == "start":
    target_page = 1
  elif action == "end":
    target_page = total_pages
  elif action == "next":
    target_page = current_page + 1
  elif action == "prev":
    target_page = current_page - 1
  else:
    target_page = current_page  # Direct jump

  target_page = max(1, min(target_page, total_pages))
  img, active_page, total, label = render_pdf_page_as_image(
      pdf_name, target_page
  )

  return img, active_page, label

SyntaxError: invalid syntax (2653788382.py, line 48)